In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [2]:
pth = os.path.join('..', '..', '..')
pth_data = os.path.join(pth, 'data')
pth_in = os.path.join(pth_data, 'development', 'modeling')
pth_feat = os.path.join(pth_data, 'features', 'biomart')
pth_out = os.path.join(pth, 'figures', 'mesencoder', 'rRNAModifications', 'expression')

In [3]:
fn = os.path.join(pth_feat, 'union.csv')
g = pd.read_csv(fn).mmusculus.drop_duplicates()

In [4]:
adata_bif = sc.read_h5ad(os.path.join(pth_in, 'bifurcation.h5ad'))
adata_bif.X = np.asarray(adata_bif.X.toarray() - adata_bif[:, g].X.mean(axis = 1))
adata_bif

AnnData object with n_obs × n_vars = 11243 × 24505
    obs: 'sample', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'celltype', 't', 'total_counts_ribo', 'pct_counts_ribo', 'T+ Sox2+', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.1', 'seurat_clusters', 'age', 'RNA_snn_res.0.5', 'ident', 'velocity_self_transition', 'celltype_original', 'source', 'GSE162534', 'GSE229103', 'rRNAModifications', 'mesencoder'
    uns: 'celltype_colors'
    obsm: 'X_feat', 'X_mesencoder', 'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_traj', 'velocity_umap'
    layers: 'logcounts', 'scaled'

In [5]:
adata = sc.read_h5ad(os.path.join(pth_in, 'mesencoder.h5ad'))
adata = adata[adata.obs.mesencoder == 'False'].copy()
adata.X = np.asarray(adata.X.toarray() - adata[:, g].X.mean(axis = 1))
adata

AnnData object with n_obs × n_vars = 1786 × 24505
    obs: 'sample', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'celltype', 't', 'total_counts_ribo', 'pct_counts_ribo', 'T+ Sox2+', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.1', 'seurat_clusters', 'age', 'RNA_snn_res.0.5', 'ident', 'velocity_self_transition', 'celltype_original', 'source', 'GSE162534', 'GSE229103', 'rRNAModifications', 'mesencoder'
    obsm: 'X_feat', 'X_mesencoder', 'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_traj', 'velocity_umap'
    layers: 'logcounts', 'scaled'

In [ ]:
pad = .05
fig, ax = plt.subplots(1, 1, figsize = (5, 5))
sc.pl.embedding(adata_bif, 'X_mesencoder', dimensions = [1, 0],
                size = 80, alpha = .5, frameon = False,
                show = False, ax = ax)
xlim = ax.get_xlim(); ylim = ax.get_ylim()
xlim = (xlim[0], xlim[1] - .375)
xpad = pad * (xlim[1] - xlim[0])
ypad = pad * (ylim[1] - ylim[0])

g_plt = {'Zeb1'   : ('green', -.25),
         'Snai1'  : ('orange', 0),
         'Twist1' : ('purple', .35)}

for var, (c, thresh) in g_plt.items():
    adata_plt = adata[adata[:, var].X > thresh].copy()
    sc.pl.embedding(adata_plt, 'X_mesencoder', dimensions = [1, 0],
                    na_color = c, size = 80, alpha = .5,
                    frameon = False, show = False, ax = ax)

sc.pl.embedding(adata, 'X_mesencoder', dimensions = [1, 0],
                color = 'celltype', size = 80, alpha = 0,
                legend_loc = 'on data', legend_fontoutline = 1,
                frameon = False, show = False, ax = ax)
ax.set_xlim(xlim[0] - xpad, xlim[1] + xpad)
ax.set_ylim(ylim[0] - ypad, ylim[1] + ypad)
ax.set_title(''); ax.invert_yaxis()

label_dict = {
    'Premigratory Neural Crest' : 'pmNC',
    'Migratory Neural Crest'    : 'mNC',
    'Neuroglial Progenitors'    : '',
    'Cranial Mesenchyme'        : 'CM'}

for label in ax.texts:
    _label_ = label.get_text()
    if _label_ in label_dict:
        label.set_text(label_dict[_label_])

fn = os.path.join(pth_out, 'rRNAModifications.pdf')
plt.savefig(fn, bbox_inches = 'tight')

In [ ]:
# pad = .05
# fig, ax = plt.subplots(1, 1, figsize = (5, 5))
# sc.pl.embedding(adata_bif, 'X_mesencoder', dimensions = [1, 0],
#                 color = 'Zeb1', cmap = 'Reds',
#                 size = 80, alpha = .5, colorbar_loc = None,
#                 frameon = False, show = False, ax = ax)
# sc.pl.embedding(adata_bif, 'X_mesencoder', dimensions = [1, 0],
#                 color = 'celltype', size = 80, alpha = 0,
#                 legend_loc = 'on data', legend_fontoutline = 1,
#                 frameon = False, show = False, ax = ax)
# ax.set_title(''); ax.invert_yaxis()

# xlim = ax.get_xlim(); ylim = ax.get_ylim()
# xlim = (xlim[0], xlim[1] - .375)
# xpad = pad * (xlim[1] - xlim[0])
# ypad = pad * (ylim[1] - ylim[0])
# ax.set_xlim(xlim[0] - xpad, xlim[1] + xpad)
# ax.set_ylim(ylim[0] - ypad, ylim[1] + ypad)

# label_dict = {
#     'Epiblast'                   : 'Epi',
#     'Lateral Plate Mesoderm'     : 'LPM',
#     'Neuromesodermal Progenitor' : 'NMP',
#     'Posterior Epiblast'         : 'pEpi',
#     'Presomitic Mesoderm'        : 'PSM'}

# for label in ax.texts:
#     _label_ = label.get_text()
#     if _label_ in label_dict:
#         label.set_text(label_dict[_label_])

# # fn = os.path.join(pth_out, 'Gast_signature.pdf')
# # plt.savefig(fn, bbox_inches = 'tight')
# plt.show()